# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic information about the dataset
print(f"Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets (@id, name)
record_sets = dataset.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"  @id: {rs.id} | name: {rs.name}")

# Show fields for each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    print("Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# ---
# You may need to update the record set IDs and field IDs based on the printed overview above.
# For this FAIR^2 dataset we'll select the main record set for protein measurements.
# Example record set id (replace with actual printed @id if different):
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/recset1'  # Placeholder

# Collect all record set ids
record_sets_ids = [rs.id for rs in dataset.record_sets]
print("Record set IDs:", record_sets_ids)

# Load records from each record set to DataFrames
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records for record set: {record_set_id}")
    dataframes[record_set_id] = df

# Show columns for first record set
if record_sets_ids:
    print(f"Columns in {record_sets_ids[0]}:")
    print(dataframes[record_sets_ids[0]].columns.tolist())
    display(dataframes[record_sets_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: analyze a numeric protein attribute (update field id as appropriate)
record_set_id = record_sets_ids[0]  # Replace with the main table
df = dataframes[record_set_id]
# Choose a numeric field, such as molecular weight (MW) if present
numeric_field_candidates = [c for c in df.columns if ('weight' in c.lower() or 'mw' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower())]
print("Possible numeric analysis fields:", numeric_field_candidates)
# Use first numeric candidate if available for demonstration
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    try:
        # Filter records based on threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field (z-score)
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by a likely categorical/textual field, e.g., 'accession' or 'description'
        group_field_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == 'object']
        print("Possible group-by fields:", group_field_candidates)
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped data by {group_field}, showing mean of {numeric_field}:")
            display(grouped_df.head())
    except Exception as e:
        print("Error during EDA:", e)
else:
    print("No numeric fields found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Example histogram of the major numeric field, if found
if numeric_field_candidates:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30, color='skyblue', edgecolor='black')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.